In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# hide
def lpc(x: np.ndarray, K: int) -> np.ndarray:
    """The coefficients h_1, ..., h_K that best predict each x[n] from x[n-1], ..., x[n-K]."""
    N = len(x)
    X = np.stack([x[K - k:N - k] for k in range(1, K + 1)], axis=1)   # row n: x[n-1], ..., x[n-K]
    h, *_ = np.linalg.lstsq(X, x[K:], rcond=None)                     # minimize the squared error
    return h

In [ ]:
def pulse_train(f0: float, duration: float, f_s: int) -> np.ndarray:
    """A train of unit impulses, f0 of them per second: a crude glottal pulse train."""
    e = np.zeros(int(duration * f_s))
    e[::round(f_s / f0)] = 1.0
    return e


vowel = pq.Audio.from_file("../assets/vowel-a.wav")   # or vowel-e, vowel-i, vowel-o, vowel-u
f_s = vowel.sample_rate
x = vowel.samples[:, 0].astype(np.float64)
h = lpc(x[f_s:f_s + f_s // 2], K=40)
coefficients = np.concatenate([[1.0], -h])

# The frequency response of the filter (the DFT of its impulse response, Chapter 9)
impulse = np.zeros(8192)
impulse[0] = 1.0
response = np.fft.rfft(lfilter([1.0], coefficients, impulse))
f = np.fft.rfftfreq(len(impulse), 1 / f_s)
plt.figure(figsize=(10, 3.5))
plt.plot(f, 20 * np.log10(np.abs(response)))
plt.xlim(0, 5000)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB)")
plt.show()

# Sing the first phrase of Twinkle, Twinkle, Little Star on this vowel
melody = [48, 48, 55, 55, 57, 57, 55, 55]
e = np.concatenate([pulse_train(pq.helper.pitch_to_frequency(pitch), 0.5, f_s)
                    for pitch in melody])
pq.play(vowel)
pq.play(pq.Audio(lfilter([1.0], coefficients, e), f_s), normalize=True)